# tt-mlir #8722 repro + fix 검증

TTMetal flatbuffer translator가 D2M의 `arith.constant`를 거부하는 버그(`Dialect 'arith' not found`).
런타임: **CPU 고RAM**. colab-8570-repro 노트북 구조 재사용, upstream/main HEAD(`70b7117e5`) 기준.


In [ ]:
!nproc
!free -h

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang ninja-build cmake git python3.12-venv libgtest-dev libgmock-dev

In [ ]:
%cd /content
!rm -rf /content/tt-mlir
!git clone --branch colab-8722 https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git log --oneline -3


In [ ]:
import os
os.environ["TTMLIR_TOOLCHAIN_DIR"] = "/opt/ttmlir-toolchain/"
!mkdir -p /opt/ttmlir-toolchain
!chown -R $(whoami) /opt/ttmlir-toolchain

In [ ]:
%cd /content/tt-mlir
!cmake -B env/build env -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++
!cmake --build env/build --parallel $(nproc)


In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake -G Ninja -B build -DCMAKE_BUILD_TYPE=Release -DTTMLIR_ENABLE_STABLEHLO=ON -DTTMLIR_ENABLE_RUNTIME=OFF -DTTMLIR_ENABLE_RUNTIME_TESTS=OFF -DTTMLIR_ENABLE_OPMODEL=OFF -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF -DCMAKE_BUILD_PARALLEL_LEVEL=$(nproc)"
!bash -c "source env/activate && cmake --build build --target ttmlir-opt ttmlir-translate -- -j$(nproc)" 2>&1 | tee /content/main_build.log | tail -150


In [ ]:
import subprocess, os
r = subprocess.run(["tail", "-n", "60", "/content/main_build.log"], capture_output=True, text=True)
print(r.stdout)
print("ttmlir-opt exists:", os.path.exists("/content/tt-mlir/build/bin/ttmlir-opt"))

In [ ]:
repro_mlir = r"""module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  ttcore.device_module {
    builtin.module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
      func.func public @main(%arg0: tensor<f64>, %arg1: tensor<f64>, %arg2: tensor<16x22xf64>, %arg3: tensor<22x18xf64>, %arg4: tensor<18x24xf64>, %arg5: tensor<16x24xf64>) -> tensor<16x24xf64> {
        %0 = \"ttir.dot_general\"(%arg2, %arg3) <{batch_dims_lhs = array<i64>, batch_dims_rhs = array<i64>, contract_dims_lhs = array<i64: 1>, contract_dims_rhs = array<i64: 0>}> : (tensor<16x22xf64>, tensor<22x18xf64>) -> tensor<16x18xf64>
        %1 = \"ttir.reshape\"(%arg0) <{shape = [1 : i32, 1 : i32]}> : (tensor<f64>) -> tensor<1x1xf64>
        %2 = \"ttir.broadcast\"(%1) <{broadcast_dimensions = array<i64: 16, 18>}> : (tensor<1x1xf64>) -> tensor<16x18xf64>
        %3 = \"ttir.multiply\"(%0, %2) : (tensor<16x18xf64>, tensor<16x18xf64>) -> tensor<16x18xf64>
        %4 = \"ttir.dot_general\"(%3, %arg4) <{batch_dims_lhs = array<i64>, batch_dims_rhs = array<i64>, contract_dims_lhs = array<i64: 1>, contract_dims_rhs = array<i64: 0>}> : (tensor<16x18xf64>, tensor<18x24xf64>) -> tensor<16x24xf64>
        %5 = \"ttir.reshape\"(%arg1) <{shape = [1 : i32, 1 : i32]}> : (tensor<f64>) -> tensor<1x1xf64>
        %6 = \"ttir.broadcast\"(%5) <{broadcast_dimensions = array<i64: 16, 24>}> : (tensor<1x1xf64>) -> tensor<16x24xf64>
        %7 = \"ttir.multiply\"(%arg5, %6) : (tensor<16x24xf64>, tensor<16x24xf64>) -> tensor<16x24xf64>
        %8 = \"ttir.add\"(%4, %7) : (tensor<16x24xf64>, tensor<16x24xf64>) -> tensor<16x24xf64>
        return %8 : tensor<16x24xf64>
      }
    }
  }
}
"""
with open("/content/repro_8722.mlir", "w") as f:
    f.write(repro_mlir)
print("written")


In [ ]:
%cd /content/tt-mlir
import subprocess
r1 = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-opt --ttir-to-ttmetal-pipeline -o /content/repro_8722.ttmetal.mlir /content/repro_8722.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("=== ttmlir-opt rc:", r1.returncode, "===")
print(r1.stdout[-2000:])
print("=== stderr ===")
print(r1.stderr[-3000:])

if r1.returncode == 0:
    out = open("/content/repro_8722.ttmetal.mlir").read()
    print("\n=== arith.constant 잔존 여부:", "arith.constant" in out, "===")
    if "arith.constant" in out:
        for line in out.splitlines():
            if "arith.constant" in line:
                print(line)
else:
    print(">>> pipeline 자체가 실패 -- 이슈의 f64 repro가 이 경로로 현재도 안 통할 수 있음, 별도 축소 필요")


In [ ]:
%cd /content/tt-mlir
import subprocess
r2 = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-translate --ttmetal-to-flatbuffer -o /content/repro_8722.ttm /content/repro_8722.ttmetal.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== ttmlir-translate (미패치) rc:", r2.returncode, "===")
print("stdout:", r2.stdout)
print("stderr:", r2.stderr)
expected = "Dialect `arith' not found"
print()
print(">>> negative control 판정:", "PASS (에러 정확히 재현됨)" if expected in r2.stderr else "FAIL (에러 문구 불일치 -- 수동 확인 필요)")


In [ ]:
%cd /content/tt-mlir
import subprocess
r3 = subprocess.run(
    ["bash", "-c", "source env/activate && build/bin/llvm-lit -v test/ttmlir/Dialect/TTIR/metal_layout_misc.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("=== rc:", r3.returncode, "===")
print(r3.stdout[-3000:])
print(r3.stderr[-1000:])


In [ ]:
%cd /content/tt-mlir
path = "lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp"
src = open(path).read()

assert '#include "mlir/Dialect/EmitC/IR/EmitC.h"' in src
src = src.replace(
    '#include "mlir/Dialect/EmitC/IR/EmitC.h"',
    '#include "mlir/Dialect/Arith/IR/Arith.h"\n#include "mlir/Dialect/EmitC/IR/EmitC.h"',
    1,
)
assert "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect," in src
src = src.replace(
    "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,",
    "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,\n"
    "                        mlir::arith::ArithDialect,",
    1,
)
open(path, "w").write(src)
print(open(path).read())


In [ ]:
%cd /content/tt-mlir
import subprocess
rb = subprocess.run(
    ["bash", "-c", "source env/activate && cmake --build build --target ttmlir-translate -- -j$(nproc)"],
    capture_output=True, text=True, timeout=600,
)
print("=== rebuild rc:", rb.returncode, "===")
print(rb.stdout[-2000:])
print(rb.stderr[-2000:])

r4 = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-translate --ttmetal-to-flatbuffer -o /content/repro_8722_fixed.ttm /content/repro_8722.ttmetal.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== ttmlir-translate (패치후) rc:", r4.returncode, "===")
print("stdout:", r4.stdout)
print("stderr:", r4.stderr)
print()
print(">>> fix 판정:", "PASS (rc=0, 번역 성공)" if r4.returncode == 0 else "FAIL -- 추가 조사 필요")
